In [26]:
import random
import re
from transformers import AutoTokenizer, AutoModelForCausalLM

In [27]:
import torch

# Check GPU availability
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory Available: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

# Load Jan model and tokenizer
tokenizer = AutoTokenizer.from_pretrained("janhq/Jan-v1-4B")
model = AutoModelForCausalLM.from_pretrained(
    "janhq/Jan-v1-4B",
    device_map="auto",  # Automatically use GPU if available
    dtype=torch.float16  # Use half precision for faster inference
)
print(f"Model loaded on: {model.device}")

Using device: cuda
GPU: NVIDIA GeForce RTX 5090
Memory Available: 33.66 GB


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Model loaded on: cuda:0


In [28]:
# Sim parameters
PRODUCTS = {
    "Red Bull": {"cost": 1.50, "price": 2.50, "inventory": 1},
    "Snickers": {"cost": 0.50, "price": 1.25, "inventory": 1}
}
DAILY_FEE = 5.0
START_BALANCE = 100.0
DAYS = 7
TRACE = []

In [29]:
def simulate_sales(product, inventory, day):
    base_sales = random.randint(2, 7)
    if day % 7 in [5, 6]:  # Weekend
        base_sales += random.randint(2, 5)
    return min(base_sales, inventory)

def extract_decision_from_reasoning(reasoning_text, state):
    """Extract actionable decisions from the model's thinking process."""
    text_lower = reasoning_text.lower()
    actions = []
    
    # Check if model recognizes the need to restock based on context
    inventory_critical = state['inventory']['Red Bull'] == 0 or state['inventory']['Snickers'] == 0
    
    # Extract restock intentions for Red Bull
    rb_patterns = [
        r"restock.*red bull.*?(\d+)",
        r"order.*red bull.*?(\d+)",
        r"buy.*red bull.*?(\d+)",
        r"get.*red bull.*?(\d+)",
        r"purchase.*red bull.*?(\d+)",
        r"red bull.*?(\d+).*?units?",
        r"(\d+).*?red bull",
    ]
    
    rb_qty = None
    for pattern in rb_patterns:
        match = re.search(pattern, text_lower)
        if match:
            rb_qty = int(match.group(1))
            break
    
    # Check for implicit restock need for Red Bull
    rb_needs_restock = any(phrase in text_lower for phrase in [
        'red bull.*out', 'red bull.*empty', 'red bull.*zero', 'red bull.*0',
        'need.*red bull', 'red bull.*low', 'restock.*red bull', 'no red bull'
    ])
    
    if rb_qty and rb_qty <= 50:
        cost = PRODUCTS['Red Bull']['cost'] * rb_qty
        if state['balance'] >= cost:
            actions.append(f"restock red bull {rb_qty}")
    elif (rb_needs_restock or state['inventory']['Red Bull'] == 0) and not rb_qty:
        # Model recognizes need but didn't specify amount - use intelligent default
        max_affordable = int(state['balance'] / PRODUCTS['Red Bull']['cost'])
        restock_qty = min(15, max_affordable)
        if restock_qty >= 5:
            actions.append(f"restock red bull {restock_qty}")
            print(f"    -> Inferred Red Bull restock need from reasoning")
    
    # Extract restock intentions for Snickers
    snickers_patterns = [
        r"restock.*snickers.*?(\d+)",
        r"order.*snickers.*?(\d+)",
        r"buy.*snickers.*?(\d+)",
        r"get.*snickers.*?(\d+)",
        r"purchase.*snickers.*?(\d+)",
        r"snickers.*?(\d+).*?units?",
        r"(\d+).*?snickers",
    ]
    
    snickers_qty = None
    for pattern in snickers_patterns:
        match = re.search(pattern, text_lower)
        if match:
            snickers_qty = int(match.group(1))
            break
    
    # Check for implicit restock need for Snickers
    snickers_needs_restock = any(phrase in text_lower for phrase in [
        'snickers.*out', 'snickers.*empty', 'snickers.*zero', 'snickers.*0',
        'need.*snickers', 'snickers.*low', 'restock.*snickers', 'no snickers'
    ])
    
    if snickers_qty and snickers_qty <= 50:
        cost = PRODUCTS['Snickers']['cost'] * snickers_qty
        if state['balance'] >= cost:
            actions.append(f"restock snickers {snickers_qty}")
    elif (snickers_needs_restock or state['inventory']['Snickers'] == 0) and not snickers_qty:
        # Model recognizes need but didn't specify amount - use intelligent default
        max_affordable = int(state['balance'] / PRODUCTS['Snickers']['cost'])
        restock_qty = min(20, max_affordable)
        if restock_qty >= 5:
            actions.append(f"restock snickers {restock_qty}")
            print(f"    -> Inferred Snickers restock need from reasoning")
    
    # Extract price change intentions for Red Bull
    rb_price_patterns = [
        r"(?:set|change|make|put|raise|lower|adjust).*red bull.*price.*to.*?\$?(\d+\.?\d*)",
        r"red bull.*price.*\$?(\d+\.?\d*)",
        r"price.*red bull.*to.*\$?(\d+\.?\d*)",
    ]
    for pattern in rb_price_patterns:
        match = re.search(pattern, text_lower)
        if match:
            price = float(match.group(1))
            if 1.0 <= price <= 10.0 and price != state['prices']['Red Bull']:
                actions.append(f"set red bull price to {price:.2f}")
                break
    
    # Extract price change intentions for Snickers
    snickers_price_patterns = [
        r"(?:set|change|make|put|raise|lower|adjust).*snickers.*price.*to.*?\$?(\d+\.?\d*)",
        r"snickers.*price.*\$?(\d+\.?\d*)",
        r"price.*snickers.*to.*\$?(\d+\.?\d*)",
    ]
    for pattern in snickers_price_patterns:
        match = re.search(pattern, text_lower)
        if match:
            price = float(match.group(1))
            if 0.5 <= price <= 10.0 and price != state['prices']['Snickers']:
                actions.append(f"set snickers price to {price:.2f}")
                break
    
    # Final fallback: if inventory is zero and no actions, force restock
    if not actions and inventory_critical:
        print(f"    -> No actions extracted but inventory critical - forcing restock")
        if state['inventory']['Red Bull'] == 0:
            max_affordable = int(state['balance'] / PRODUCTS['Red Bull']['cost'])
            if max_affordable >= 5:
                actions.append(f"restock red bull {min(10, max_affordable)}")
        
        if state['inventory']['Snickers'] == 0:
            max_affordable = int(state['balance'] / PRODUCTS['Snickers']['cost'])
            if max_affordable >= 5:
                actions.append(f"restock snickers {min(15, max_affordable)}")
    
    return ", ".join(actions) if actions else "no action"

def agent_decision(state):
    user_msg = (
        f"You are managing a vending machine business. Current state:\n"
        f"- Balance: ${state['balance']:.2f}\n"
        f"- Inventory: Red Bull={state['inventory']['Red Bull']}, Snickers={state['inventory']['Snickers']}\n"
        f"- Prices: Red Bull=${state['prices']['Red Bull']:.2f}, Snickers=${state['prices']['Snickers']:.2f}\n"
        f"- Yesterday's sales: Red Bull={state['sales']['Red Bull']}, Snickers={state['sales']['Snickers']}\n"
        f"- Daily operating fee: ${DAILY_FEE}\n\n"
        f"Product costs: Red Bull=${PRODUCTS['Red Bull']['cost']:.2f}, Snickers=${PRODUCTS['Snickers']['cost']:.2f}\n\n"
        "What should you do today? Consider:\n"
        "1. Should you restock any products? How many?\n"
        "2. Should you adjust prices based on sales?\n"
        "3. Do you have enough balance to restock?\n\n"
        "Think through your decision carefully."
    )
    messages = [
        {"role": "user", "content": user_msg},
    ]
    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
    )
    
    # Move inputs to GPU (model.device handles both CPU and GPU cases)
    inputs = {k: v.to(model.device) for k, v in inputs.items()}
    
    # Generate on GPU with optimized settings
    with torch.no_grad():  # Disable gradient computation for inference
        outputs = model.generate(
            **inputs, 
            max_new_tokens=200,
            pad_token_id=tokenizer.eos_token_id,
            temperature=0.7,
            do_sample=True,
            top_p=0.9,
            eos_token_id=tokenizer.eos_token_id
        )
    
    response = tokenizer.decode(outputs[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True)
    
    # Debug: print raw response (first 400 chars to see more reasoning)
    print(f"  [DEBUG] Model reasoning (first 400 chars): {response[:400]}...")
    
    # Extract decision intent from the full reasoning (not truncated)
    decision = extract_decision_from_reasoning(response, state)
    print(f"  [EXTRACTED] Decision: {decision}")
    
    return decision

def parse_and_execute_action(agent_action, state):
    """Parse agent action and return what needs to be done at start of next day."""
    restock_orders = {}
    price_updates = {}
    
    for product in PRODUCTS:
        product_lower = product.lower()
        
        # Parse restock orders
        restock_match = re.search(rf"restock {product_lower} (\d+)", agent_action.lower())
        if restock_match:
            qty = int(restock_match.group(1))
            cost = PRODUCTS[product]["cost"] * qty
            if state["balance"] >= cost:
                restock_orders[product] = qty
            else:
                restock_orders[product] = 0
        else:
            restock_orders[product] = 0
        
        # Parse price changes
        price_match = re.search(rf"set {product_lower} price to ([\d.]+)", agent_action.lower())
        if price_match:
            new_price = float(price_match.group(1))
            price_updates[product] = new_price
    
    return restock_orders, price_updates

def run_simulation():
    state = {
        "balance": START_BALANCE,
        "inventory": {k: v["inventory"] for k, v in PRODUCTS.items()},
        "prices": {k: v["price"] for k, v in PRODUCTS.items()},
        "sales": {k: 0 for k in PRODUCTS.keys()}
    }
    TRACE = []
    
    pending_restocks = {}
    pending_price_changes = {}

    for day in range(1, DAYS + 1):
        print(f"\n=== DAY {day} ===")
        
        # Execute previous day's orders at start of day
        if pending_restocks:
            print(f"  Executing restocks from previous day...")
            for product, qty in pending_restocks.items():
                if qty > 0:
                    cost = PRODUCTS[product]["cost"] * qty
                    state["inventory"][product] += qty
                    state["balance"] -= cost
                    print(f"    -> Restocked {qty} {product} for ${cost:.2f}")
        
        if pending_price_changes:
            print(f"  Applying price changes from previous day...")
            for product, new_price in pending_price_changes.items():
                old_price = state["prices"][product]
                state["prices"][product] = new_price
                print(f"    -> {product} price: ${old_price:.2f} -> ${new_price:.2f}")
        
        # Simulate sales for the day
        day_sales = {}
        for product in PRODUCTS:
            sold = simulate_sales(product, state["inventory"][product], day)
            state["inventory"][product] -= sold
            state["balance"] += sold * state["prices"][product]
            day_sales[product] = sold

        # Pay daily fee
        state["balance"] -= DAILY_FEE

        # Agent makes decision for tomorrow
        agent_action = agent_decision({
            "balance": state["balance"],
            "inventory": state["inventory"],
            "sales": day_sales,
            "prices": state["prices"]
        })
        
        # Parse the action for execution tomorrow
        pending_restocks, pending_price_changes = parse_and_execute_action(agent_action, state)

        # Log trace with end-of-day state
        trace_entry = {
            "day": day,
            "balance": state["balance"],
            "inventory": state["inventory"].copy(),
            "sales": day_sales.copy(),
            "prices": state["prices"].copy(),
            "agent_action": agent_action,
            "pending_restock": pending_restocks.copy(),
            "pending_price_changes": pending_price_changes.copy() if pending_price_changes else None
        }

        TRACE.append(trace_entry)

        if state["balance"] < 0:
            print(f"Bankrupt on day {day}")
            break

    return TRACE

In [30]:
if __name__ == "__main__":
    trace = run_simulation()
    for day in trace:
        print(f"Day {day['day']}:")
        print(f"  Balance: ${day['balance']:.2f}")
        print(f"  Inventory: {day['inventory']}")
        print(f"  Prices: {day['prices']}")
        print(f"  Sales: {day['sales']}")
        print(f"  Agent Action: {day['agent_action']}")
        print(f"  Pending Restock (for tomorrow): {day.get('pending_restock', {})}")
        if day.get("pending_price_changes"):
            print(f"  Pending Price Changes (for tomorrow): {day['pending_price_changes']}")
        print("-" * 40)


=== DAY 1 ===
  [DEBUG] Model reasoning (first 400 chars): <think>
Okay, let me try to figure out what I should do today for my vending machine business. So, I'm starting with a balance of $98.75. The inventory is currently zero for both Red Bull and Snickers. That's a problem because if there are no products, I can't sell anything. Wait, but yesterday's sales show I sold one Red Bull and one Snickers. So maybe I restocked before that? Hmm, maybe the inve...
    -> No actions extracted but inventory critical - forcing restock
  [EXTRACTED] Decision: restock red bull 10, restock snickers 15

=== DAY 2 ===
  Executing restocks from previous day...
    -> Restocked 10 Red Bull for $15.00
    -> Restocked 15 Snickers for $7.50
  [DEBUG] Model reasoning (first 400 chars): <think>
Okay, let me try to figure out what I should do today for the vending machine business. So, I'm the manager, and I need to check the current state and make decisions based on the given information.

First, let me 